# Notebook 08: Circuit Tracing & Attribution Graphs

**Prerequisites**: SAEs (nb03), Activation Patching (nb04)

This is the current frontier of mechanistic interpretability -- Anthropic's 2025 circuit tracing methodology. Let's dig in.

## Section 1: From Patching to Attribution Graphs

**Circuit tracing** ([Anthropic, January 2025](https://transformer-circuits.pub/2025/attribution-graphs/methods.html)) is a major advance over activation patching. While patching tells you *which* components matter, circuit tracing reveals the full **computational graph** -- how information flows step-by-step through the model to produce a specific output.

The key innovation: combine **Sparse Autoencoders** (to identify features) with **attribution** (to trace information flow between features across layers). You get an "attribution graph" -- a directed graph where:
- Nodes = SAE features (interpretable units)
- Edges = causal influence (how much one feature contributes to another)

This lets you answer: "What is the step-by-step computation the model performs to produce this output?"

## Section 2: The Methodology

Here's what Anthropic's circuit tracing pipeline looks like:

1. **Train SAEs** on model activations — not just the residual stream, but also sublayer outputs (attention and MLP) — to capture features at every stage of computation (or use pretrained ones like Gemma Scope)
2. **Replace activations** with SAE reconstructions throughout the model (the "replacement model")
3. For a given prompt and output token, compute **attribution scores** between features:
   - How much does feature F_i at layer L contribute to feature F_j at any downstream layer?
   - This uses **activation $\times$ gradient attribution**: $a_i \cdot \partial f_j / \partial f_i$, i.e., the product of the upstream feature activation $a_i$ and the downstream gradient
4. **Threshold and prune** to get a sparse graph of the most important connections
5. **Interpret** the graph: each node (feature) has a human-readable description; edges show information flow

The key insight: SAE features serve as a "vocabulary" of interpretable concepts. The attribution graph shows how these concepts compose to produce behavior.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Conceptual illustration: we'll simulate what an attribution graph looks like
# In practice, this requires SAEs trained on every layer + the full attribution pipeline

# Simulated attribution graph for: "The Eiffel Tower is in" -> " Paris"
# Based on the kind of circuits found in Anthropic's 2025 papers

nodes = {
    # Layer 0-2: Token-level features
    "L1: 'Eiffel'_token": (0, 3),
    "L1: 'Tower'_token": (0, 2),
    "L2: 'is_in'_syntax": (0, 1),
    
    # Layer 3-5: Entity features
    "L4: Eiffel_Tower_entity": (1, 2.5),
    "L4: location_query": (1, 1),
    
    # Layer 6-8: Knowledge features  
    "L7: Paris_associated": (2, 3),
    "L7: France_context": (2, 2),
    "L7: European_capital": (2, 1),
    
    # Layer 9-11: Output features
    "L10: city_name_output": (3, 2),
    "L11: ' Paris'_prediction": (4, 2),
}

edges = [
    ("L1: 'Eiffel'_token", "L4: Eiffel_Tower_entity", 0.8),
    ("L1: 'Tower'_token", "L4: Eiffel_Tower_entity", 0.6),
    ("L2: 'is_in'_syntax", "L4: location_query", 0.7),
    ("L4: Eiffel_Tower_entity", "L7: Paris_associated", 0.9),
    ("L4: Eiffel_Tower_entity", "L7: France_context", 0.5),
    ("L4: location_query", "L7: European_capital", 0.6),
    ("L7: Paris_associated", "L10: city_name_output", 0.85),
    ("L7: France_context", "L10: city_name_output", 0.3),
    ("L7: European_capital", "L10: city_name_output", 0.4),
    ("L10: city_name_output", "L11: ' Paris'_prediction", 0.95),
]

# Plot
fig, ax = plt.subplots(figsize=(14, 8))

# Draw edges
for src, dst, weight in edges:
    x1, y1 = nodes[src]
    x2, y2 = nodes[dst]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=weight*3, color=plt.cm.Blues(weight), alpha=0.7))

# Draw nodes
for name, (x, y) in nodes.items():
    color = ['#FFB3BA', '#BAFFC9', '#BAE1FF', '#FFFFBA', '#E8BAFF'][int(x)]
    ax.plot(x, y, 'o', markersize=20, color=color, markeredgecolor='black', markeredgewidth=1.5)
    # Short label
    short_name = name.split(": ")[1] if ": " in name else name
    ax.annotate(short_name, (x, y), fontsize=7, ha='center', va='bottom',
                xytext=(0, 12), textcoords='offset points')

ax.set_xlim(-0.5, 4.5)
ax.set_ylim(0, 4)
ax.set_xlabel("Processing Stage (Layer Groups)")
ax.set_title("Conceptual Attribution Graph: 'The Eiffel Tower is in' -> ' Paris'", fontsize=13)

# Layer group labels
for i, label in enumerate(["Tokens", "Entities", "Knowledge", "Output", "Prediction"]):
    ax.text(i, -0.3, label, ha='center', fontsize=9, style='italic')

ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

## Section 3: Key Findings from "On the Biology of a Large Language Model"

Anthropic's companion paper ([Biology of LLMs](https://transformer-circuits.pub/2025/attribution-graphs/biology.html)) applied circuit tracing to Claude 3.5 Sonnet. Here's what they found:

### Multi-step Reasoning
For prompts requiring reasoning, the attribution graph shows intermediate "thinking" features. The model doesn't jump from input to output -- it constructs intermediate representations.

### Planning Behavior
On tasks like writing a poem with specific constraints, the model activates constraint-tracking features *before* generating each word. It "plans ahead" through feature activations.

### Multilingual Features
Many features are **language-agnostic** -- the same "Eiffel Tower" feature activates whether the input is in English, French, or Chinese. The model has abstract conceptual representations that transcend language.

### Safety-Relevant Circuits
Features related to refusal, honesty, and safety were identified. These form circuits that detect potentially harmful requests and route the model toward appropriate responses.

### Hallucination Mechanisms
When the model hallucinates, the attribution graph shows knowledge features failing to activate, and the model falls back to pattern-matching features. This points to potential interventions.

## Section 4: Practical Circuit Tracing (with available tools)

Anthropic's full pipeline isn't open-source yet, but we can approximate parts of it with available tools. The building blocks are:
1. Pretrained SAEs (from SAELens or Gemma Scope)
2. Attribution via gradients (similar to attribution patching)
3. Feature visualization (via Neuronpedia)

In [ ]:
# Approximate feature-level attribution using SAEs + gradients
# This demonstrates the concept using our GPT-2 setup
# Full circuit tracing on production models requires significant compute

from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()

prompt = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(prompt)
token_strs = [model.tokenizer.decode(t) for t in tokens[0]]

# Get the model's prediction
logits, cache = model.run_with_cache(prompt)
predicted_token = logits[0, -1].argmax().item()
predicted_str = model.to_string([predicted_token])
print(f"Prompt: '{prompt}'")
print(f"Predicted next token: '{predicted_str}'")

In [ ]:
# Compute gradient of predicted token logit w.r.t. residual stream at each layer
# This tells us which positions and layers contribute most to the prediction

# We need a fresh forward pass with gradient tracking via hooks
model.zero_grad()

attribution_by_layer = {}

def save_grad_hook(activation, hook):
    activation.retain_grad()
    attribution_by_layer[hook.name] = activation
    return activation

hooks = [(f"blocks.{layer}.hook_resid_post", save_grad_hook) for layer in range(model.cfg.n_layers)]
logits = model.run_with_hooks(prompt, fwd_hooks=hooks)
target_logit = logits[0, -1, predicted_token]

# Backward pass
target_logit.backward()

# Compute attribution magnitude at each layer and position
attr_matrix = torch.zeros(model.cfg.n_layers, len(token_strs))
for layer in range(model.cfg.n_layers):
    resid = attribution_by_layer[f"blocks.{layer}.hook_resid_post"]
    grad = resid.grad[0]  # (seq, d_model)
    act = resid.detach()[0]
    # Attribution ~ activation * gradient (element-wise, summed over d_model)
    attr = (act * grad).sum(dim=-1).abs().cpu()
    attr_matrix[layer] = attr

plt.figure(figsize=(14, 8))
plt.imshow(attr_matrix.numpy(), cmap="Reds", aspect="auto")
plt.colorbar(label="Attribution magnitude")
plt.xlabel("Token Position")
plt.ylabel("Layer")
plt.title(f"Gradient Attribution: which (layer, position) contributes to predicting '{predicted_str}'")
plt.xticks(range(len(token_strs)), token_strs, rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

# Highlight top contributors
flat = attr_matrix.flatten()
top_k = 5
top_indices = flat.argsort(descending=True)[:top_k]
print(f"\nTop {top_k} (layer, token) attributions for predicting '{predicted_str}':")
for idx in top_indices:
    l = idx.item() // len(token_strs)
    p = idx.item() % len(token_strs)
    print(f"  Layer {l}, token '{token_strs[p]}': {attr_matrix[l, p]:.4f}")

## Section 5: The Future — Full Computational Graphs

Circuit tracing is the current frontier, and it's evolving fast. Key directions:

1. **Open-source reproduction**: The community is working on reproducing Anthropic's results on open models (GPT-2, Gemma, Llama). Expect tools to emerge throughout 2025.

2. **Automated circuit discovery**: Combining circuit tracing with automated interpretability (using LLMs to label features) to fully automate the pipeline from model -> computational graph -> human-readable explanation.

3. **Intervention via circuits**: Once you have the computational graph, you can intervene at specific nodes — more precise than steering vectors (nb07), which operate on raw activation space.

4. **Safety monitoring**: Real-time monitoring of which circuits are active during inference, flagging concerning patterns (deception circuits, harmful content circuits).

5. **Model comparison**: Comparing attribution graphs across models to understand how capabilities emerge with scale.

**The vision**: A future where every model prediction can be traced to a human-readable computational graph, making AI behavior fully auditable.

---
### Running Example: IOI — Attribution Graph Sketch

This is part of our **running example** investigating how GPT-2-small handles the Indirect Object Identification (IOI) task across all techniques in this guide.

**The task**: "When Mary and John went to the store, John gave a drink to" — the model should predict "Mary".

The gradient attribution analysis demonstrated above in Section 4 already illustrates the core idea behind circuit tracing applied to a factual recall prompt. The IOI task follows the same pattern: we can use gradient-based attribution to identify which (layer, token) positions contribute most to predicting "Mary". The attribution graph for IOI would show information flowing from the "Mary" token position through duplicate token heads, S-inhibition heads, and finally name mover heads to the output.

**Connection to the full running example**: Notebook 04 identified the IOI circuit's components via activation patching. Here, attribution tracing shows the same information flow from a complementary angle — instead of asking "which components matter?" (patching), we ask "how does information flow between components?" (attribution). In a full circuit tracing setup with SAEs (as in Anthropic's 2025 work), each node in the graph would be a named, interpretable feature rather than a raw (layer, position) pair.

## Exercises

### Exercise 1: Build a Mini Attribution Graph

For a simple prompt like `"The capital of France is"`, compute a basic attribution graph: measure how much each attention head at each layer contributes to the final logit of `" Paris"`. Use gradient-based attribution (grad * activation) to build a (layer x head) attribution matrix, then visualize it as a heatmap.

**Goal**: Understand which heads are most responsible for the factual recall prediction.

<details>
<summary>Hint</summary>

Run a forward pass with hooks to capture each head's output activation (use `hook_result` which has shape `[batch, seq, head, d_model]`). Then compute the gradient of the `" Paris"` logit with respect to each head's output. The attribution score for each head is `(gradient * activation).sum()`. You'll need to call `.retain_grad()` on the hooked activations and then call `.backward()` on the target logit.

</details>

In [ ]:
prompt = "The capital of France is"
target = " Paris"

# Get target token id
target_id = model.to_single_token(target)

# Run forward pass with hooks to capture each head's output
head_activations = {}

def save_head_hook(activation, hook):
    activation.retain_grad()
    head_activations[hook.name] = activation
    return activation

hooks = [(f"blocks.{layer}.attn.hook_result", save_head_hook) for layer in range(model.cfg.n_layers)]
logits = model.run_with_hooks(prompt, fwd_hooks=hooks)

# Compute gradients w.r.t. the target logit
target_logit = logits[0, -1, target_id]
target_logit.backward()

# Build attribution matrix (layer x head)
# Attribution = (gradient * activation).sum() for each head at the final token position
attr_matrix = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)
for layer in range(model.cfg.n_layers):
    act = head_activations[f"blocks.{layer}.attn.hook_result"]
    grad = act.grad[0, -1]  # (n_heads, d_model)
    val = act.detach()[0, -1]  # (n_heads, d_model)
    attr_matrix[layer] = (grad * val).sum(dim=-1).abs().cpu()

# TODO: Try modifying this!
# Visualize as heatmap
plt.figure(figsize=(14, 8))
plt.imshow(attr_matrix.detach().numpy(), cmap="Reds", aspect="auto")
plt.colorbar(label="Attribution magnitude")
plt.xlabel("Head")
plt.ylabel("Layer")
plt.title(f"Head Attribution for predicting '{target}' given '{prompt}'")
plt.xticks(range(model.cfg.n_heads))
plt.yticks(range(model.cfg.n_layers))
plt.tight_layout()
plt.show()

# Print top contributing heads
flat = attr_matrix.flatten()
top_k = 10
top_indices = flat.argsort(descending=True)[:top_k]
print(f"\nTop {top_k} heads for predicting '{target}':")
for idx in top_indices:
    l = idx.item() // model.cfg.n_heads
    h = idx.item() % model.cfg.n_heads
    print(f"  L{l}H{h}: attribution = {attr_matrix[l, h]:.4f}")

### Exercise 2: Compare Direct vs Indirect Effects

For the same prompt (`"The capital of France is"`), separate each attention head's contribution into:
- **Direct effect**: the head's output projected directly through the unembedding matrix W_U to the target logit.
- **Indirect effect**: the head's influence on later components (total effect minus direct effect). Use activation patching (zero ablation) to measure total effect.

Which heads have large indirect effects (i.e., they matter mostly because they influence downstream heads)?

<details>
<summary>Hint</summary>

**Direct effect**: For each head, take its output vector at the final token position and project through W_U to get the logit contribution: `direct = head_output @ model.W_U[:, target_id]`. **Total effect**: Zero-ablate each head one at a time and measure the change in the target logit. **Indirect effect** = total effect - direct effect. Heads with large indirect but small direct effects are "upstream" heads that set up information for later heads.

</details>

In [ ]:
# Compute direct effect: each head's output projected through W_U
prompt = "The capital of France is"
target_id = model.to_single_token(" Paris")
logits, cache = model.run_with_cache(prompt)
baseline_logit = logits[0, -1, target_id].item()

direct_effects = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)
for layer in range(model.cfg.n_layers):
    head_out = cache[f"blocks.{layer}.attn.hook_result"][0, -1]  # (n_heads, d_model)
    for head in range(model.cfg.n_heads):
        direct_effects[layer, head] = (head_out[head] @ model.W_U[:, target_id]).item()

# Compute total effect: zero-ablation patching each head
total_effects = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        def zero_hook(activation, hook, h=head):
            activation[:, :, h, :] = 0
            return activation
        patched_logits = model.run_with_hooks(prompt, fwd_hooks=[(f"blocks.{layer}.attn.hook_result", zero_hook)])
        total_effects[layer, head] = baseline_logit - patched_logits[0, -1, target_id].item()

# Indirect effect = total - direct
indirect_effects = total_effects - direct_effects

# TODO: Try modifying this!
# Plot direct, indirect, and total effects side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, data, title in zip(axes,
                           [direct_effects.detach().numpy(), indirect_effects.detach().numpy(), total_effects.detach().numpy()],
                           ["Direct Effect", "Indirect Effect", "Total Effect"]):
    im = ax.imshow(data, cmap="RdBu", aspect="auto", vmin=-np.abs(data).max(), vmax=np.abs(data).max())
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_title(title)
    ax.set_xticks(range(model.cfg.n_heads))
    ax.set_yticks(range(model.cfg.n_layers))
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f"Direct vs Indirect Effects for predicting ' Paris'", fontsize=14)
plt.tight_layout()
plt.show()

# Print heads with largest indirect effects
print("Heads with largest |indirect effect| (upstream heads):")
flat = indirect_effects.abs().flatten()
top_k = 5
top_indices = flat.argsort(descending=True)[:top_k]
for idx in top_indices:
    l = idx.item() // model.cfg.n_heads
    h = idx.item() % model.cfg.n_heads
    print(f"  L{l}H{h}: direct={direct_effects[l,h]:.3f}, indirect={indirect_effects[l,h]:.3f}, total={total_effects[l,h]:.3f}")

## Section 6: Key Takeaways & Further Reading

**What you should remember:**
- Circuit tracing combines SAEs (features) + attribution (flow) into full computational graphs
- The "replacement model" approach uses SAE reconstructions throughout the forward pass
- Attribution graphs reveal multi-step reasoning, planning, and knowledge retrieval
- Safety-relevant circuits can be identified and potentially monitored
- This is the current frontier -- tools are still being developed

**How is this different from activation patching (nb04)?**
- Patching identifies important components; circuit tracing shows how they connect
- Patching works at the neuron/head level; circuit tracing works at the feature level (via SAEs)
- Circuit tracing produces interpretable graphs; patching produces importance scores

**Further reading:**
- [Circuit Tracing: Revealing Computational Graphs in Language Models](https://transformer-circuits.pub/2025/attribution-graphs/methods.html) (Anthropic, 2025) -- the methods paper
- [On the Biology of a Large Language Model](https://transformer-circuits.pub/2025/attribution-graphs/biology.html) (Anthropic, 2025) -- the findings paper
- [Interpreting Jailbreaks and Prompt Injections with Attribution Graphs](https://labs.zenity.io/p/interpreting-jailbreaks-and-prompt-injections-with-attribution-graphs) -- safety applications
- [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) -- the SAE foundation this builds on

**Next**: Notebook 09 -- Tools & Practice (hands-on with the interpretability toolkit)